# Rust ownership: short experiments

Polars is fast partly because its engine is written in Rust. This notebook is a set of
small experiments with the three rules that make Rust different from Python:
values are immutable by default, a value has one owner, and you either borrow or move it.

### 1. Loops look like Python

In [8]:
let hours = [40, 13, 45, 50, 40];   // a fixed-size array
for h in hours {
    println!("{h} h/week");
}

40 h/week
13 h/week
45 h/week
50 h/week
40 h/week


()

### 2. `let` is immutable, `let mut` can change
A counter inside a loop has to be `mut`. A cutoff that never changes does not.

In [9]:
let cutoff = 40;
let mut overtime = 0;
for h in hours {
    if h > cutoff {
        overtime += 1;
    }
}
println!("{overtime} of {} people work more than {cutoff} h/week", hours.len());

2 of 5 people work more than 40 h/week


*Fails on purpose*: assigning to a plain `let` twice.

In [10]:
let cutoff = 40;
cutoff = 35;
println!("{cutoff}");

Error: cannot assign twice to immutable variable `cutoff`

Error: value assigned to `cutoff` is never read

The error says `cannot assign twice to immutable variable` and suggests `mut`. Python would silently allow this.

### 3. A `Vec` has one owner: `let b = a` moves it
An array of numbers above was copied on each use. A `Vec` lives on the heap and is *moved*, not copied.

In [11]:
let hours = vec![40, 13, 45, 50, 40];
let moved = hours;               // ownership goes to moved
println!("{moved:?}");

[40, 13, 45, 50, 40]


*Fails on purpose*: using the old name after the move.

In [12]:
let hours = vec![40, 13, 45, 50, 40];
let moved = hours;
println!("{hours:?}");           // `hours` no longer owns anything

Error: borrow of moved value: `hours`

Fix: `.clone()` makes a real second list (costs memory), or `&` borrows without taking ownership.

In [ ]:
let hours = vec![40, 13, 45, 50, 40];
let copy = hours.clone();
{                                // the notebook kernel needs borrows inside a block
    let view = &hours;           // a borrow: no copy, hours still the owner
    println!("owner {hours:?}\ncopy  {copy:?}\nview  {view:?}");
}

owner [40, 13, 45, 50, 40]
copy  [40, 13, 45, 50, 40]
view  [40, 13, 45, 50, 40]


()

### 4. `for h in vec` moves the vec, `for h in &vec` borrows it

In [14]:
let hours = vec![40, 13, 45, 50, 40];
let mut total = 0;
for h in &hours {                // borrow, so hours survives the loop
    total += h;
}
println!("total {total}, average {:.1}, still have {hours:?}", total as f64 / hours.len() as f64);

total 188, average 37.6, still have [40, 13, 45, 50, 40]


*Fails on purpose*: looping without `&` moves the vec into the loop, so it is gone afterwards.

In [15]:
let hours = vec![40, 13, 45, 50, 40];
for h in hours { let _ = h; }
println!("{hours:?}");

Error: borrow of moved value: `hours`

### 5. Many readers or one writer, never both
*Fails on purpose*: pushing to the vec while a loop is reading it. Python allows this and can skip elements.

In [16]:
let mut hours = vec![40, 13, 45, 50, 40];
for h in &hours {
    if *h > 40 {
        hours.push(0);           // write while the loop still holds a read borrow
    }
}

Error: cannot borrow `hours` as mutable because it is also borrowed as immutable

Fix: finish reading first, then write. Here the loop collects into a new vec, and `retain` is the safe way to drop elements in place.

In [17]:
let mut hours = vec![40, 13, 45, 50, 40];
let mut over: Vec<i32> = Vec::new();
for h in &hours {
    if *h > 40 { over.push(*h); }
}                                // read borrow ends here
hours.retain(|&h| h <= 40);      // now a write is allowed
println!("overtime {over:?}, remaining {hours:?}");

overtime [45, 50], remaining [40, 13, 40]


### What I take from this

* `let` is frozen, `let mut` is not. The compiler, not a code review, enforces it.
* A `Vec` has exactly one owner. Assigning or looping over it by value moves it; `&` lends it.
* Reading and writing the same data at the same time is a compile error, not a silent bug.

These checks happen at compile time, so the running program pays nothing for them.
That is why a Rust engine like Polars can be both strict and fast while I keep writing Python.